# V6.5 — Camber Offload Launcher

Bản này giải quyết triệt để lỗi `No space left on device` bằng cách **không xử lý dataset lớn trên Kaggle nữa**.

Kaggle chỉ làm 3 việc nhỏ:

1. lấy `CAMBER_API_KEY`;
2. upload 2 worker script rất nhỏ lên Camber Stash;
3. submit Camber CPU jobs nếu SDK `camber.mpi` có sẵn, hoặc in sẵn lệnh/prompt để chạy trong Camber Nova.

Toàn bộ download/extract/staging nặng sẽ chạy trên **Camber compute**, rồi output được ghi trực tiếp về Camber Stash.

## Hai job

### Job A — External source staging
Không cần V6.2E:
- Open Images V7: Balloon / Kite / Plastic bag / Bird / Person
- MPCD
- Power Equipment Image Dataset
- tạo manifests / source overlays / provenance
- upload từng source lên Stash rồi xóa temp local trên node

### Job B — Parent background pool
Chỉ Camber compute mới chạm V6.2E archive:
- đọc V6.2E manifest
- stream base/overlay tar
- chỉ materialize một pool nhỏ TRAIN có insulator
- đóng gói background pool để dùng synthetic composition
- upload pool lên Stash
- cleanup temp

Như vậy Kaggle không còn phải giữ base tar, overlay tar, extraction tree hay runtime copy nữa.

In [ ]:
# 0. CONFIG + CAMBER AUTH
from pathlib import Path
import os, sys, json, shutil, subprocess, textwrap

STAGE_ROOT = "stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage"
JOB_ROOT = f"{STAGE_ROOT}/jobs"

# CPU-only staging. SMALL = 16 vCPU / 64GB RAM in current Camber docs.
NODE_SIZE = "SMALL"

# Kaggle secret
key = os.environ.get("CAMBER_API_KEY")
if not key:
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("CAMBER_API_KEY")
    except Exception:
        key = None

if not key:
    raise RuntimeError("CAMBER_API_KEY chưa bật trong Kaggle Secrets.")

os.environ["CAMBER_API_KEY"] = key

camber_bin = Path.home() / ".camber/bin/camber"
if shutil.which("camber") is None and not camber_bin.exists():
    subprocess.check_call(
        "curl -sL https://cli.cambercloud.com/install-v2.sh | bash",
        shell=True
    )

os.environ["PATH"] = f"{Path.home()}/.camber/bin:{Path.home()}/.local/bin:" + os.environ.get("PATH","")
CAMBER = shutil.which("camber") or str(camber_bin)

assert Path(CAMBER).exists(), "Camber CLI not found"
print("CAMBER:", CAMBER)

me = subprocess.run([CAMBER, "me"], text=True, capture_output=True, env=os.environ.copy())
print(me.stdout or me.stderr)

In [ ]:
# 1. WORKER A — EXTERNAL SOURCES ONLY
EXTERNAL_WORKER = r"""
from pathlib import Path
import os, sys, json, hashlib, shutil, subprocess, zipfile, tarfile, random, xml.etree.ElementTree as ET
import pandas as pd
import numpy as np

ROOT = Path("/tmp/v65_external_worker")
SRC = ROOT / "sources"
OUT = ROOT / "out"
for p in [SRC, OUT]:
    p.mkdir(parents=True, exist_ok=True)

STAGE_ROOT = "stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage"
OI_CLASSES = ["Balloon", "Kite", "Plastic bag", "Bird", "Person"]
OI_MAX = 180
SEED = 20261007
MPCD_GDRIVE_ID = "1KyciMmwL2_p_-mwckHFqG5fkBA1jzATv"
HF_REPO = "sxiong/Power-equipment-image-dataset"
HF_FILE = "transmission line object detection.zip"

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "pandas", "numpy", "pillow", "pyyaml", "gdown",
    "huggingface_hub", "fiftyone==1.22.1"
])

from PIL import Image
import yaml, gdown
from huggingface_hub import hf_hub_download
import fiftyone.utils.openimages as fouo

CAMBER = shutil.which("camber") or str(Path.home()/".camber/bin/camber")
assert Path(CAMBER).exists(), "Camber CLI missing on worker node"

def run(cmd):
    r = subprocess.run(cmd, text=True, capture_output=True, env=os.environ.copy())
    if r.returncode != 0:
        raise RuntimeError((r.stderr or r.stdout)[:5000])
    return r

def put(local, remote):
    run([CAMBER, "stash", "cp", str(local), str(remote)])

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        while True:
            b=f.read(1024*1024)
            if not b: break
            h.update(b)
    return h.hexdigest()

def norm_names(names):
    if isinstance(names, dict):
        return [names[k] for k in sorted(names, key=lambda x:int(x))]
    return list(names or [])

def read_yolo(lp):
    rows=[]
    if not Path(lp).exists(): return rows
    for line in Path(lp).read_text(errors="ignore").splitlines():
        p=line.split()
        if len(p)<5: continue
        try: rows.append((int(float(p[0])), *map(float,p[1:5])))
        except: pass
    return rows

def write_yolo(lp, rows):
    Path(lp).parent.mkdir(parents=True, exist_ok=True)
    txt="\n".join(f"{int(c)} {x:.8f} {y:.8f} {w:.8f} {h:.8f}" for c,x,y,w,h in rows if w>0 and h>0)
    Path(lp).write_text(txt + ("\n" if txt else ""))

def xyxy_to_yolo(box):
    x1,y1,x2,y2=box
    return [(x1+x2)/2,(y1+y2)/2,max(0,x2-x1),max(0,y2-y1)]

manifest = []

# -------------------------------------------------------------------
# A1. Open Images partial TRAIN download
# -------------------------------------------------------------------
oi_root = SRC/"open_images_v7"
oi_root.mkdir(parents=True, exist_ok=True)
boxable=set(fouo.get_classes(version="v7"))
segable=set(fouo.get_segmentation_classes(version="v7"))

for cls in OI_CLASSES:
    assert cls in boxable, cls
    label_types=["detections"]
    if cls in segable:
        label_types.append("segmentations")

    fouo.download_open_images_split(
        dataset_dir=str(oi_root),
        split="train",
        version="v7",
        label_types=label_types,
        classes=[cls],
        shuffle=True,
        seed=SEED,
        max_samples=OI_MAX,
        num_workers=8,
    )

# package raw partial OI source immediately to stash, then retain only source folder
oi_tar = OUT/"open_images_v7_partial.tar.gz"
with tarfile.open(oi_tar,"w:gz") as t:
    t.add(oi_root, arcname="open_images_v7")
put(oi_tar, f"{STAGE_ROOT}/open_images/{oi_tar.name}")

manifest.append({
    "source":"OpenImagesV7",
    "role":"foreign_object_appearance_source",
    "classes":OI_CLASSES,
    "split":"train",
    "archive":f"{STAGE_ROOT}/open_images/{oi_tar.name}",
})
oi_tar.unlink(missing_ok=True)

# -------------------------------------------------------------------
# A2. MPCD
# -------------------------------------------------------------------
mpcd_zip = SRC/"MPCD.zip"
url=f"https://drive.google.com/uc?id={MPCD_GDRIVE_ID}"
res=gdown.download(url, str(mpcd_zip), quiet=False)
if res is None:
    raise RuntimeError("MPCD Google Drive download failed")

mpcd_dir=SRC/"MPCD"
mpcd_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(mpcd_zip) as z:
    z.extractall(mpcd_dir)
mpcd_zip.unlink(missing_ok=True)

# identify candidate detection YAMLs and map broken/fracture/discontinuity -> class 5
mapped_root=OUT/"mpcd_broken_strand"
for s in ["train","external_eval"]:
    (mapped_root/"images"/s).mkdir(parents=True, exist_ok=True)
    (mapped_root/"labels"/s).mkdir(parents=True, exist_ok=True)

mpcd_rows=[]
for yp in mpcd_dir.rglob("data.yaml"):
    try:
        cfg=yaml.safe_load(yp.read_text())
        names=norm_names(cfg.get("names"))
    except:
        continue

    broken_ids=[]
    for cid,n in enumerate(names):
        z=str(n).lower().replace("_"," ").replace("-"," ")
        if any(k in z for k in ["broken","fracture","fractured","discontinu","strand break"]):
            broken_ids.append(cid)
    if not broken_ids:
        continue

    base=Path(cfg.get("path",yp.parent))
    if not base.is_absolute(): base=(yp.parent/base).resolve()

    for sk in ["train","val","test"]:
        raw=cfg.get(sk)
        if raw is None: continue
        p=Path(raw)
        if not p.is_absolute(): p=base/p

        if p.is_file():
            imgs=[]
            for x in p.read_text().splitlines():
                if not x.strip(): continue
                q=Path(x.strip())
                imgs.append(q if q.is_absolute() else base/q)
        elif p.exists():
            imgs=[x for x in p.rglob("*") if x.suffix.lower() in {".jpg",".jpeg",".png",".webp",".bmp"}]
        else:
            continue

        for ip in imgs:
            s=str(ip).replace("\\","/")
            if "/images/" not in s: continue
            lp=Path(s.replace("/images/","/labels/")).with_suffix(".txt")
            if not lp.exists(): continue

            rows=[(5,x,y,w,h) for c,x,y,w,h in read_yolo(lp) if c in broken_ids]
            if not rows: continue

            target="train" if sk=="train" else "external_eval"
            uid=sha256_file(ip)[:16]
            dst_i=mapped_root/"images"/target/f"mpcd_{uid}{ip.suffix.lower()}"
            dst_l=mapped_root/"labels"/target/f"mpcd_{uid}.txt"
            shutil.copy2(ip,dst_i)
            write_yolo(dst_l,rows)
            mpcd_rows.append({
                "source":"MPCD","source_split":sk,"target_split":target,
                "source_image":str(ip),"source_sha256":sha256_file(ip),
                "boxes":len(rows),
            })

pd.DataFrame(mpcd_rows).to_csv(mapped_root/"manifest.csv",index=False)

mpcd_tar=OUT/"mpcd_broken_strand_stage.tar.gz"
with tarfile.open(mpcd_tar,"w:gz") as t:
    t.add(mapped_root,arcname="mpcd_broken_strand")
put(mpcd_tar,f"{STAGE_ROOT}/mpcd/{mpcd_tar.name}")
manifest.append({
    "source":"MPCD",
    "role":"broken_strand_direct_powerline",
    "mapped_images":len(mpcd_rows),
    "archive":f"{STAGE_ROOT}/mpcd/{mpcd_tar.name}",
})
mpcd_tar.unlink(missing_ok=True)
shutil.rmtree(mpcd_dir,ignore_errors=True)

# -------------------------------------------------------------------
# A3. Power Equipment object-detection subset
# -------------------------------------------------------------------
hf_path=Path(hf_hub_download(
    repo_id=HF_REPO,
    filename=HF_FILE,
    repo_type="dataset",
    cache_dir=str(SRC/"hf_cache"),
))
pe_dir=SRC/"power_equipment"
pe_dir.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(hf_path) as z:
    z.extractall(pe_dir)
shutil.rmtree(SRC/"hf_cache",ignore_errors=True)

pe_stage=OUT/"power_equipment_stage"
for s in ["train","external_eval"]:
    (pe_stage/"images"/s).mkdir(parents=True,exist_ok=True)
    (pe_stage/"labels"/s).mkdir(parents=True,exist_ok=True)

imgs_by_stem={}
for p in pe_dir.rglob("*"):
    if p.suffix.lower() in {".jpg",".jpeg",".png",".bmp",".webp"}:
        imgs_by_stem.setdefault(p.stem,[]).append(p)

mapping={"dx_dg":5,"yw":4,"nw":3}
review_only={"dx_sg":"strand_loosening_review_only"}
pe_rows=[]
review_rows=[]

for xp in pe_dir.rglob("*.xml"):
    try:
        root=ET.parse(xp).getroot()
    except:
        continue

    filename=root.findtext("filename")
    ip=None
    if filename:
        q=xp.parent/filename
        if q.exists(): ip=q
        elif imgs_by_stem.get(Path(filename).stem): ip=imgs_by_stem[Path(filename).stem][0]
    elif imgs_by_stem.get(xp.stem):
        ip=imgs_by_stem[xp.stem][0]
    if ip is None: continue

    size=root.find("size")
    if size is None:
        with Image.open(ip) as im: W,H=im.size
    else:
        W=int(float(size.findtext("width"))); H=int(float(size.findtext("height")))

    rows=[]
    for obj in root.findall("object"):
        name=(obj.findtext("name") or "").strip()
        bb=obj.find("bndbox")
        if bb is None: continue
        xmin=float(bb.findtext("xmin"))/W
        ymin=float(bb.findtext("ymin"))/H
        xmax=float(bb.findtext("xmax"))/W
        ymax=float(bb.findtext("ymax"))/H
        box=[max(0,min(1,xmin)),max(0,min(1,ymin)),max(0,min(1,xmax)),max(0,min(1,ymax))]

        if name in mapping:
            xc,yc,w,h=xyxy_to_yolo(box)
            rows.append((mapping[name],xc,yc,w,h))
        if name in review_only:
            review_rows.append({
                "source_image":str(ip),"source_xml":str(xp),
                "source_class":name,"role":review_only[name],
                "bbox_xyxy":json.dumps(box),
            })

    if not rows: continue

    frac=int(hashlib.sha256(str(ip).encode()).hexdigest()[:12],16)/float(16**12)
    target="train" if frac<0.80 else "external_eval"
    uid=sha256_file(ip)[:16]
    dst_i=pe_stage/"images"/target/f"pe_{uid}{ip.suffix.lower()}"
    dst_l=pe_stage/"labels"/target/f"pe_{uid}.txt"
    shutil.copy2(ip,dst_i)
    write_yolo(dst_l,rows)
    pe_rows.append({
        "source":"PowerEquipment","target_split":target,
        "source_image":str(ip),"source_sha256":sha256_file(ip),
        "mapped_classes":",".join(map(str,sorted(set(r[0] for r in rows)))),
        "boxes":len(rows),
    })

pd.DataFrame(pe_rows).to_csv(pe_stage/"manifest.csv",index=False)
pd.DataFrame(review_rows).to_csv(pe_stage/"dx_sg_review_only.csv",index=False)

pe_tar=OUT/"power_equipment_stage.tar.gz"
with tarfile.open(pe_tar,"w:gz") as t:
    t.add(pe_stage,arcname="power_equipment_stage")
put(pe_tar,f"{STAGE_ROOT}/power_equipment/{pe_tar.name}")

manifest.append({
    "source":"PowerEquipment",
    "role":"direct_powerline",
    "mapping":{"dx_dg":"broken_strand","yw":"foreign_object","nw":"bird_nest","dx_sg":"review_only"},
    "mapped_images":len(pe_rows),
    "archive":f"{STAGE_ROOT}/power_equipment/{pe_tar.name}",
})
pe_tar.unlink(missing_ok=True)
shutil.rmtree(pe_dir,ignore_errors=True)

# -------------------------------------------------------------------
# Final source manifest
# -------------------------------------------------------------------
final_manifest={
    "stage":"v6p5_external_sources",
    "sources":manifest,
    "policy":{
        "open_images_full_image_direct_merge":False,
        "open_images_split":"train",
        "MPCD_train_to_train":True,
        "MPCD_val_test_to_external_eval":True,
        "PowerEquipment_dx_sg_direct_merge":False,
        "test_modified":False,
    }
}
mp=OUT/"external_stage_manifest.json"
mp.write_text(json.dumps(final_manifest,indent=2))
put(mp,f"{STAGE_ROOT}/external_stage_manifest.json")

print(json.dumps(final_manifest,indent=2))
print("DONE External Stage")
"""
print("External worker chars:", len(EXTERNAL_WORKER))

In [ ]:
# 2. WORKER B — BUILD SMALL INTERNAL TRAIN BACKGROUND POOL ON CAMBER
PARENT_WORKER = r"""
from pathlib import Path
import os, sys, json, hashlib, shutil, subprocess, zipfile, tarfile, io
import pandas as pd

subprocess.check_call([sys.executable,"-m","pip","install","-q","pandas","pillow"])

from PIL import Image

ROOT=Path("/tmp/v65_parent_bg_worker")
CACHE=ROOT/"cache"
POOL=ROOT/"pool"
CACHE.mkdir(parents=True,exist_ok=True)
(POOL/"images").mkdir(parents=True,exist_ok=True)
(POOL/"labels").mkdir(parents=True,exist_ok=True)

CAMBER=shutil.which("camber") or str(Path.home()/".camber/bin/camber")
assert Path(CAMBER).exists()

V6ROOT="stash://bunpmc/projects/yolov11sdi/datasets"
TAG="v6p2e_23562a45b343"
STAGE="stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage"
MAX_BG=800
SEED="20261007"

def run(cmd):
    r=subprocess.run(cmd,text=True,capture_output=True,env=os.environ.copy())
    if r.returncode!=0: raise RuntimeError((r.stderr or r.stdout)[:5000])
    return r

def get(remote,local):
    local=Path(local); local.parent.mkdir(parents=True,exist_ok=True)
    if not local.exists():
        run([CAMBER,"stash","cp",remote,str(local)])
    return local

def put(local,remote):
    run([CAMBER,"stash","cp",str(local),remote])

freeze_path=get(f"{V6ROOT}/v6p2e/{TAG}/v6p2e.freeze.json",CACHE/"freeze.json")
freeze=json.loads(freeze_path.read_text())

base_tar=get(f"{V6ROOT}/curated_v6_rs1280_{freeze['base_tag']}.tar",CACHE/"base.tar")
overlay_tar=get(
    f"{V6ROOT}/v6p1/{freeze['parent_dataset_tag']}/v6p1_fotl_overlay_{freeze['overlay_tag']}.tar.gz",
    CACHE/"overlay.tar.gz"
)
labels_zip=get(f"{V6ROOT}/v6p2e/{TAG}/v6p2e_reconciled_labels.zip",CACHE/"labels.zip")
manifest_csv=get(f"{V6ROOT}/v6p2e/{TAG}/v6p2e_split_manifest.csv",CACHE/"manifest.csv")

manifest=pd.read_csv(manifest_csv)

bt=tarfile.open(base_tar,"r:*")
ot=tarfile.open(overlay_tar,"r:*")
rz=zipfile.ZipFile(labels_zip,"r")

def build_idx(tf):
    idx={}
    for m in tf.getmembers():
        if not m.isfile(): continue
        n=m.name.replace("\\","/").lstrip("./")
        pi=n.find("images/"); pl=n.find("labels/")
        key=n[pi:] if pi>=0 else (n[pl:] if pl>=0 else None)
        if key is not None and key not in idx: idx[key]=m
    return idx

bidx=build_idx(bt); oidx=build_idx(ot)
ridx={Path(n).name:n for n in rz.namelist() if n.endswith(".txt")}

def source_keys(row):
    op=str(row["image_path_source"]).replace("\\","/")
    osp=str(row["previous_v6p1_split"])
    marker=f"/images/{osp}/"
    suffix=op.split(marker,1)[1]
    if suffix.startswith("base/"):
        src="base"; rel=suffix[len("base/"):]
    elif suffix.startswith("fotl/"):
        src="fotl"; rel=suffix[len("fotl/"):]
    else:
        raise RuntimeError(suffix)
    imgk=f"images/{osp}/{rel}"
    lblk=f"labels/{osp}/{Path(rel).with_suffix('.txt').as_posix()}"
    return src,rel,imgk,lblk

def label_text(row):
    rn=f"{row['sha256']}.txt"
    if str(row.get("resolution","")).startswith("merged") and rn in ridx:
        return rz.read(ridx[rn]).decode("utf-8",errors="ignore")
    src,rel,imgk,lblk=source_keys(row)
    tf,idx=(bt,bidx) if src=="base" else (ot,oidx)
    m=idx.get(lblk)
    if m is None: raise KeyError(lblk)
    return tf.extractfile(m).read().decode("utf-8",errors="ignore")

def image_bytes(row):
    src,rel,imgk,lblk=source_keys(row)
    tf,idx=(bt,bidx) if src=="base" else (ot,oidx)
    m=idx.get(imgk)
    if m is None: raise KeyError(imgk)
    return tf.extractfile(m).read(), Path(rel).suffix.lower()

def has_insulator(txt):
    for line in txt.splitlines():
        p=line.split()
        if len(p)>=5:
            try:
                if int(float(p[0]))==0: return True
            except: pass
    return False

tr=manifest[manifest["split"].astype(str)=="train"].copy()
tr["_order"]=tr["sha256"].astype(str).map(lambda x:hashlib.sha256((x+SEED).encode()).hexdigest())
tr=tr.sort_values("_order")

rows=[]
for idx,row in tr.iterrows():
    if len(rows)>=MAX_BG: break
    try:
        txt=label_text(row)
        if not has_insulator(txt): continue
        raw,suf=image_bytes(row)
        name=f"{str(row['sha256'])[:16]}_{idx:06d}{suf or '.jpg'}"
        ip=POOL/"images"/name
        lp=POOL/"labels"/Path(name).with_suffix(".txt")
        ip.write_bytes(raw)
        lp.write_text(txt)
        rows.append({
            "sha256":str(row["sha256"]),
            "manifest_index":int(idx),
            "source_path":str(row["image_path_source"]),
            "image":f"images/{name}",
            "label":f"labels/{Path(name).with_suffix('.txt').name}",
        })
    except Exception:
        continue

pd.DataFrame(rows).to_csv(POOL/"manifest.csv",index=False)
assert len(rows)>=300, f"Too few backgrounds: {len(rows)}"

bt.close(); ot.close(); rz.close()

out=ROOT/"v6p2e_train_background_pool.tar.gz"
with tarfile.open(out,"w:gz") as t:
    t.add(POOL,arcname="v6p2e_train_background_pool")

put(out,f"{STAGE}/parent_background_pool/{out.name}")

meta=ROOT/"parent_background_pool.json"
meta.write_text(json.dumps({
    "source_dataset":"V6.2E",
    "source_tag":TAG,
    "split":"train",
    "selection":"contains class 0 insulator",
    "count":len(rows),
    "archive":f"{STAGE}/parent_background_pool/{out.name}",
},indent=2))
put(meta,f"{STAGE}/parent_background_pool/{meta.name}")

# clean huge archives on worker node
shutil.rmtree(ROOT,ignore_errors=True)
print("DONE Parent Background Pool")
"""
print("Parent worker chars:", len(PARENT_WORKER))

In [ ]:
# 3. WRITE + UPLOAD WORKERS TO CAMBER STASH
local = Path("/kaggle/working/v65_camber_workers")
local.mkdir(parents=True, exist_ok=True)

external_py = local / "v65_external_stage_worker.py"
parent_py = local / "v65_parent_bg_worker.py"

external_py.write_text(EXTERNAL_WORKER)
parent_py.write_text(PARENT_WORKER)

# Compile before upload
compile(EXTERNAL_WORKER, str(external_py), "exec")
compile(PARENT_WORKER, str(parent_py), "exec")

def stash_put(local_path, remote):
    r = subprocess.run(
        [CAMBER, "stash", "cp", str(local_path), remote],
        text=True, capture_output=True, env=os.environ.copy()
    )
    if r.returncode != 0:
        raise RuntimeError(r.stderr or r.stdout)
    print("uploaded:", remote)

REMOTE_EXTERNAL = f"{JOB_ROOT}/{external_py.name}"
REMOTE_PARENT = f"{JOB_ROOT}/{parent_py.name}"

stash_put(external_py, REMOTE_EXTERNAL)
stash_put(parent_py, REMOTE_PARENT)

print("Workers ready on Camber:")
print(" ", REMOTE_EXTERNAL)
print(" ", REMOTE_PARENT)

In [ ]:
# 4. TRY TO SUBMIT CAMBER COMPUTE JOBS
#
# Camber docs expose:
#   camber.mpi.create_job(command=..., node_size="SMALL", num_nodes=1, with_gpu=False)
#
# Some Kaggle environments may not ship the Python SDK even though the CLI works.
# This cell never fails the notebook: if SDK is absent, it prints exact Nova prompts.

commands = {
    "external": (
        "bash -lc 'mkdir -p /tmp/v65job && "
        f"camber stash cp {REMOTE_EXTERNAL} /tmp/v65job/external.py && "
        "python /tmp/v65job/external.py'"
    ),
    "parent_bg": (
        "bash -lc 'mkdir -p /tmp/v65job && "
        f"camber stash cp {REMOTE_PARENT} /tmp/v65job/parent_bg.py && "
        "python /tmp/v65job/parent_bg.py'"
    ),
}

submitted = {}

try:
    import camber
    mpi = getattr(camber, "mpi", None)

    if mpi is None:
        raise ImportError("camber.mpi is not exposed in this Kaggle environment")

    for name, cmd in commands.items():
        print("Submitting:", name)
        job = mpi.create_job(
            command=cmd,
            node_size=NODE_SIZE,
            num_nodes=1,
            with_gpu=False,
        )
        submitted[name] = str(job)
        print(name, "->", job)

except Exception as e:
    print("Camber SDK submit unavailable from this Kaggle kernel:")
    print(" ", repr(e))
    print()
    print("Workers are already uploaded to Stash.")
    print("Open Camber Nova and run these two requests:")
    print()
    print(
        f"1) Run {REMOTE_EXTERNAL} on a {NODE_SIZE} CPU node. "
        "Execute it as a Python script and let it write outputs back to Camber Stash."
    )
    print(
        f"2) Run {REMOTE_PARENT} on a {NODE_SIZE} CPU node. "
        "Execute it as a Python script and let it write outputs back to Camber Stash."
    )

print("\nCommands:")
print(json.dumps(commands, indent=2))

In [ ]:
# 5. CHECK CAMBER STASH OUTPUTS
# Run this cell later; it lists server-side results without downloading them to Kaggle.

targets = [
    STAGE_ROOT,
    f"{STAGE_ROOT}/open_images",
    f"{STAGE_ROOT}/mpcd",
    f"{STAGE_ROOT}/power_equipment",
    f"{STAGE_ROOT}/parent_background_pool",
]

for target in targets:
    print("\n###", target)
    r = subprocess.run(
        [CAMBER, "stash", "ls", "-r", target],
        text=True, capture_output=True, env=os.environ.copy()
    )
    print((r.stdout or r.stderr)[:12000])

## Khi hai Camber jobs xong

Kết quả vẫn nằm trên Stash, không kéo xuống Kaggle:

`stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage/`

Dự kiến có:

- `open_images/open_images_v7_partial.tar.gz`
- `mpcd/mpcd_broken_strand_stage.tar.gz`
- `power_equipment/power_equipment_stage.tar.gz`
- `parent_background_pool/v6p2e_train_background_pool.tar.gz`
- `external_stage_manifest.json`

Đây là **staging layer**, chưa promotion vào canonical V6.5.

Bước tiếp theo sẽ chạy synthetic composition + cross-source dedup cũng trên Camber, rồi chỉ trả về manifest/QA nhỏ cho Kaggle nếu cần.

### Vì sao cách này hết lỗi disk

Kaggle không còn download:
- V6.2E base tar
- V6.2E overlay tar
- MPCD
- Power Equipment ZIP
- Open Images partial source

Kaggle chỉ giữ 2 worker script vài chục KB. Phần nặng chạy trên Camber compute và output được cất vào Stash.